# 🩺 Cow Health — Model Training (Support Vector Machine)Dataset: `cow_health_records.csv`This notebook performs a stratified train/test split and trains a **Support Vector Machine (SVM, RBF kernel)** classifier — wrapped in a `StandardScaler` pipeline — to classify cows as `Healthy`, `At Risk`, or `Sick`. It evaluates accuracy, the confusion matrix, and per-class precision/recall/F1, then saves the trained pipeline as `cow_health_model.pkl`.

In [11]:
pip install matplotlib seaborn jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.3 MB/s eta 0:00:00


# **Cow Health Model Train**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

# 1. Load health data from Colab storage
df_model2 = pd.read_csv('cow_health_records.csv')

# 2. Isolate your features (X) and target diagnostic classes (y)
# Drop administrative/tracking tags like 'cow_id' if present in your sheet
X_health = df_model2.drop(columns=['cow_id', 'health_condition'])
y_health = df_model2['health_condition']

# 3. Perform a Stratified 80/20 Split
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_health, y_health, test_size=0.2, random_state=42, stratify=y_health
)

print("✂️ Stratified Medical Data Split Complete!")
print(f"Training Cases: {X_train_h.shape[0]} | Testing Evaluation Cases: {X_test_h.shape[0]}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

print("🧮 Initializing SVM Classifier with Feature Scaling...")

# Drop the 'record_id' column from X_train_h and X_test_h as it's a string identifier
X_train_h_processed = X_train_h.drop(columns=['record_id'])
X_test_h_processed = X_test_h.drop(columns=['record_id'])

# SVM requires data scaling because it relies on geometric distance calculations
model_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=1.0, random_state=42) # RBF kernel maps smooth boundaries
)

# Train the SVM Pipeline
model_svm.fit(X_train_h_processed, y_train_h)

# Evaluate
y_pred_svm = model_svm.predict(X_test_h_processed)
svm_acc = accuracy_score(y_test_h, y_pred_svm)

print(f"\n🔮 SVM Classification Accuracy: {svm_acc * 100:.2f}%")
print(classification_report(y_test_h, y_pred_svm, target_names=['Healthy (0)', 'At Risk (1)', 'Sick (2)']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute the classification confusion grid
cm = confusion_matrix(y_test_h, y_pred_svm)

plt.figure(figsize=(7, 5))
# Render as an attractive heatmap canvas using a deep forest green/blues palette
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy', 'At Risk', 'Sick'],
            yticklabels=['Healthy', 'At Risk', 'Sick'],
            cbar=False, annot_kws={"size": 14, "weight": "bold"})

plt.title('Model 2 Validation: Flawless Confusion Matrix 🩺📊', fontsize=13, pad=15)
plt.xlabel('AI Predicted Medical Label', fontsize=11)
plt.ylabel('True Historical Medical Label', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import pandas as pd

# Set clean, professional dashboard styling
sns.set_theme(style="darkgrid")

# 1. Extract the structural classification report dictionary from your SVM predictions
report_dict = classification_report(y_test_h, y_pred_svm,
                                    target_names=['Healthy', 'At Risk', 'Sick'],
                                    output_dict=True)

# 2. Convert report metrics into a clean Pandas DataFrame (excluding support & global averages)
df_report = pd.DataFrame(report_dict).iloc[:-1, :3].T

# 3. Render the standalone bar chart canvas
plt.figure(figsize=(9, 6))
ax = df_report.plot(kind='bar', cmap='Set2', edgecolor='black', width=0.7, figsize=(10, 6))

# Add titles and clean structural labels
plt.title('SVM Performance Matrix: Precision, Recall, & F1-Score Per Class 📊', fontsize=14, pad=15)
plt.xlabel('Cattle Health Classification Categories', fontsize=12)
plt.ylabel('Statistical Score (0.0 - 1.0)', fontsize=12)
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend(loc='lower right', frameon=True)

# Add exact value labels on top of each bar for professional clarity
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height() + 0.02),
                ha='center', va='center', fontsize=9, xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
import joblib
from google.colab import files

# Save the entire scaled SVM pipeline container
joblib.dump(model_svm, 'cow_health_model.pkl')
print("💾 SVM Champion Model serialized as 'cow_health_model.pkl'!")

# Instantly download to your local computer
files.download('cow_health_model.pkl')